In [ ]:
import polars as pl
import os
import pandas as pd
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/2019-Oct.csv'
print(f"Размер файла: {os.path.getsize(file_path) / (1024**3):.2f} GB")

Mounted at /content/drive
Размер файла: 5.28 GB


In [ ]:
#Polars для предварительного чтения и фильтрации

lf = pl.scan_csv(file_path, ignore_errors=True)

session_df = (
    lf
    .select([
        "event_time", "event_type", "category_code", "brand",
        "price", "user_id", "user_session"
    ])

    .filter(
        (pl.col("price") > 0) &
        (pl.col("event_type").is_in(["view", "cart", "purchase"])) &
        (pl.col("brand").is_not_null() | pl.col("category_code").is_not_null()) &
        (pl.col("event_time") >= "2019-10-15") &
        (pl.col("event_time") < "2019-10-29")
    )
    # Парсинг даты
    .with_columns(
        pl.col("event_time")
          .str.slice(0, 19)
          .str.to_datetime(format="%Y-%m-%d %H:%M:%S")
          .alias("event_dt")
    )
    # Признаки для анализа
    .with_columns(
        (pl.col("event_type") == "view").cast(pl.Int8).alias("is_view"),
        (pl.col("event_type") == "cart").cast(pl.Int8).alias("is_cart"),
        (pl.col("event_type") == "purchase").cast(pl.Int8).alias("is_purchase"),
        pl.col("event_dt").dt.hour().alias("hour"),
        pl.col("event_dt").dt.weekday().alias("weekday"),
        (pl.col("event_dt").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
        pl.col("category_code")
          .str.split(".")
          .list.get(0)
          .alias("main_category")
    )
    # Агрегация по сессиям
    .group_by(["user_session", "user_id"])
    .agg(
        pl.col("weekday").first().alias("weekday"),
        pl.col("is_weekend").first().alias("is_weekend"),
        pl.col("hour").min().alias("start_hour"),
        pl.len().alias("events_count"),
        pl.col("is_view").sum().alias("views"),
        pl.col("is_cart").sum().alias("carts"),
        pl.col("is_purchase").sum().alias("purchases"),
        (pl.col("is_purchase").sum() > 0).cast(pl.Int8).alias("made_purchase"),
        (pl.col("is_cart").sum() > 0).cast(pl.Int8).alias("added_to_cart"),
        pl.col("price").mean().alias("avg_price"),
        pl.col("price").max().alias("max_price"),
        pl.col("main_category")
          .filter(pl.col("is_view") == 1)
          .mode()
          .first()
          .alias("main_category_viewed")
    )
    # Материализация
    .collect(engine="streaming")
)

# Конвертация в Pandas
df_pd = session_df.to_pandas()

print(f"✅ Сессий: {len(df_pd):,}")
print(f"✅ Конверсия: {df_pd['made_purchase'].mean():.3%}")
print(f"✅ Размер Pandas: {df_pd.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nПропуски:\n{df_pd.isnull().sum()}")

✅ Сессий: 4,156,287
✅ Конверсия: 6.847%
✅ Размер Pandas: 761.4 MB

Пропуски:
user_session                  1
user_id                       0
weekday                       0
is_weekend                    0
start_hour                    0
events_count                  0
views                         0
carts                         0
purchases                     0
made_purchase                 0
added_to_cart                 0
avg_price                     0
max_price                     0
main_category_viewed    1064841
dtype: int64


In [ ]:
# Копируем файл на Google Drive
!cp /content/sessions_14days.csv /content/drive/MyDrive/sessions_14days.csv

print("✅ Сохранено на Google Drive!")
print("Путь: MyDrive/sessions_14days.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Сохранено на Google Drive!
Путь: MyDrive/sessions_14days.csv
